# Notebook 01 — Data Generation and Cleaning

This notebook walks through the full synthetic data generation pipeline and validates data quality.

**Tables generated:** teams, games, weather, promotions, attendance, ticket_segments, ticket_sales, concessions, merchandise, fan_profiles, fan_transactions, model_predictions

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import DATA_SIMULATED_DIR, DATA_PROCESSED_DIR

pd.set_option('display.max_columns', 30)
print('Setup complete')

## 1. Generate All Datasets

In [ ]:
from src.simulate_business_data import main as generate_data
generate_data()

## 2. Validate and Explore Generated Data

In [ ]:
games = pd.read_csv(DATA_SIMULATED_DIR / 'games.csv', parse_dates=['game_date'])
attendance = pd.read_csv(DATA_SIMULATED_DIR / 'attendance.csv')
ticket_sales = pd.read_csv(DATA_SIMULATED_DIR / 'ticket_sales.csv')
concessions = pd.read_csv(DATA_SIMULATED_DIR / 'concessions.csv')
merchandise = pd.read_csv(DATA_SIMULATED_DIR / 'merchandise.csv')
fan_profiles = pd.read_csv(DATA_SIMULATED_DIR / 'fan_profiles.csv')
fan_transactions = pd.read_csv(DATA_SIMULATED_DIR / 'fan_transactions.csv')

print('Row counts:')
for name, df in [('games', games), ('attendance', attendance), ('ticket_sales', ticket_sales),
                  ('concessions', concessions), ('merchandise', merchandise),
                  ('fan_profiles', fan_profiles), ('fan_transactions', fan_transactions)]:
    print(f'  {name}: {len(df):,} rows')

In [ ]:
print('Games sample:')
display(games.head(5))
print('\nAttendance stats:')
display(attendance[['actual_attendance','capacity_pct','attendance_tier']].describe())

In [ ]:
# Validate no negative revenue
assert (ticket_sales['net_ticket_revenue'] >= 0).all(), 'Negative ticket revenue found!'
assert (concessions['gross_revenue'] >= 0).all(), 'Negative concession revenue found!'
assert (merchandise['gross_revenue'] >= 0).all(), 'Negative merchandise revenue found!'
# Validate attendance <= capacity
assert (attendance['actual_attendance'] <= attendance['arena_capacity']).all(), 'Attendance exceeds capacity!'
# Validate tickets_sold <= available
assert (ticket_sales['tickets_sold'] <= ticket_sales['tickets_available']).all(), 'Tickets sold exceeds available!'
print('All validation checks passed.')

## 3. Attendance Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(attendance['capacity_pct'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Attendance Capacity % Distribution')
axes[0].set_xlabel('Capacity %')
tier_counts = attendance['attendance_tier'].value_counts()
axes[1].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%', colors=['#ff9999','#ffd166','#06d6a0','#0068c9'])
axes[1].set_title('Attendance Tier Breakdown')
plt.tight_layout()
plt.show()

## 4. Clean Data

In [ ]:
from src.data_cleaning import main as clean_data
clean_data()

In [ ]:
import os
processed = list(DATA_PROCESSED_DIR.glob('*.csv'))
print(f'Processed files: {len(processed)}')
for f in processed:
    df = pd.read_csv(f)
    print(f'  {f.name}: {len(df):,} rows, {len(df.columns)} cols')